In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from tqdm import tqdm
import numpy as np
import pynapple as nap
import pandas as pd


from Utils.load_files import get_interval_pairs
from Utils.json_tools import read_formatted_json
from Utils.tuning_curve_utils import get_exposure_timestamps, tuning_curve
file_names = read_formatted_json("./file_names.json")
session_info_filename: str = file_names["session_info_filename"]
head_direction_filename: str = file_names["head_direction_filename"]
interval_table_filename: str = file_names["interval_table_filename"]
kilosort_info_filename: str = file_names["kilosort_info_filename"]


In [4]:
base_dir = r"/mnt/senzailab/Kai/#Recording/m18"

date: str | int = "260817"
multi_recording: bool = True
num_of_rec_list: list[int] = [5, ]
headplate_name: str = 'hp4'
probe_name: str = "A"

phase_key = "baseline"
num_of_bins_in_hd: int = 180
num_shuffle: int = 1000
shuffle_seed: int = 0

camera_input_channel: int = 1
camera_ttl_threshold: int = 14000
camera_ttl_active_high: bool = True

for num_of_rec in num_of_rec_list:
    with tqdm(total=5, desc="Reading files", unit="%",
              bar_format="{l_bar}{bar}| {n}/{total} [{percentage:3.0f}%]") as pbar:
        subfolder_filler = f"{date}_{num_of_rec}"
        base_dir = f"{base_dir}/{date}/{subfolder_filler}" if multi_recording else f"{base_dir}/{date}"

        data_dir: str = f"{base_dir}/data"
        kilosort_dir = next((Path(base_dir) / "kilosort" / f"Probe{probe_name}").glob("kilosort_*"))

        session_info: dict = read_formatted_json(f"{data_dir}/{session_info_filename}.json")["session_info"]
        pbar.update(1)

        interval_table = pd.read_csv(f"{data_dir}/{interval_table_filename}.csv")
        interval_pairs_all = np.asarray(
            get_interval_pairs(interval_table,
                               phase_key=phase_key), dtype=float)
        pbar.update(1)

        hd_content = read_formatted_json(f"{data_dir}/processed/{head_direction_filename}.json")
        hd_raw = hd_content[headplate_name].get('head_direction_deg')
        hd = np.asarray(hd_raw, dtype=float)
        hd = hd % 360
        hd_frames = np.asarray(hd_content[headplate_name].get('frames'), dtype=int)
        pbar.update(1)

        exposure_timestamps, adc_time_origin_s, ttl_qc = get_exposure_timestamps(
            session_info=session_info,
            camera_input_channel=camera_input_channel,
            camera_ttl_threshold=camera_ttl_threshold,
            camera_ttl_active_high=camera_ttl_active_high,
        )
        pbar.update(1)

        assert np.array_equal(hd_frames, np.arange(len(hd_frames))), (
            "Motive frame IDs must be zero-based and continuous."
        )
        assert len(hd_frames) == len(exposure_timestamps), (
            f"Motive frame/TTL count mismatch: {len(hd_frames)} frames != "
            f"{len(exposure_timestamps)} exposure pulses. No tuning_curves.tc was written."
        )
        assert len(hd) == len(hd_frames)

        # head_direction.json must already use the GUI convention: 0 degrees up,
        # positive counter-clockwise. This notebook only applies modulo 360.
        HD_tsd = nap.Tsd(t=exposure_timestamps, d=hd)
        ttl_qc["motive_frame_count"] = int(len(hd_frames))
        pbar.update(1)

    interval_pairs = interval_pairs_all[[0]]

    hd_tuning_curves_A_formatted = tuning_curve(base_dir=base_dir,
                                                kilosort_dir=kilosort_dir,
                                                probe_name=probe_name,
                                                session_info=session_info,
                                                interval_pairs=interval_pairs,
                                                HD_tsd=HD_tsd,
                                                adc_time_origin_s=adc_time_origin_s,
                                                num_of_bins_in_hd=num_of_bins_in_hd,
                                                num_shuffle=num_shuffle,
                                                shuffle_seed=shuffle_seed,
                                                is_save=True,
                                                metadata={
                                                    "epoch": phase_key,
                                                    "headplate": headplate_name,
                                                    "ttl_qc": ttl_qc,
                                                },
                                                )


Classifying HD cells: 100%|██████████| 272/272 [00:25<00:00, 10.48unit/s]
